<a href="https://colab.research.google.com/github/annaphuongwit/14_LLMs/blob/main/4_evaluation_KE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 4: Evaluating Your RAG System

This guide provides instructions for the next critical step: quantitatively evaluating the performance of your RAG application. This will allow you to measure the quality of your chatbot's answers and understand its strengths and weaknesses.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

-----

## 1.&nbsp; Why We Evaluate

So far, you've a working RAG chatbot, but how good is it? Answering this question with just a few trial conversations is difficult. Evaluation provides a systematic way to measure performance using objective metrics.

A good evaluation framework allows us to answer questions like:

* **Faithfulness**: Is the chatbot's answer factually based on the information in our documents, or is it making things up (hallucinating)?
* **Relevance**: Is the answer actually relevant to the user's question?
* **Context Quality**: Is the retrieval part of our system finding the most useful text chunks from our documents?

By the end of this guide, you'll have a repeatable process for testing your RAG system and generating a report of its performance.

-----

## 2.&nbsp; Library Installation

For evaluation, we need a new library. Our evaluation will use `ragas`, a powerful framework designed specifically for assessing RAG pipelines.

In your VSCode terminal, run the following commands:

In [ ]:
# !conda install conda-forge::ragas # ragas = 0.3.0 or 0.3.5

-----

## 3.&nbsp; Create the Evaluation Directory

It is good practice to keep our evaluation code separate from our main application code. Let's create a new directory to house our evaluation scripts.

In your VSCode file explorer, create a new folder named `evaluation` in the root of your project (at the same level as `src` and `data`).

-----

## 4.&nbsp; Building the Evaluation Assets

Our evaluation requires two key assets: a set of questions to test the system and a configuration file for our evaluation settings.

### 4.1 Create the Evaluation Questions

To test our system, we need a consistent set of questions and corresponding "ground truth" answers. The ground truth is the ideal, factually correct answer we expect the RAG system to produce.

1.  In the `evaluation` directory, create a new file named `evaluation_questions.py`.
2.  Add the following list to this file. This defines our test dataset.

In [ ]:

EVAL_DATA: list[dict[str, str]] = [
    {
        "question": "Who is Alice and what is her sister doing at the beginning of the story?",
        "ground_truth": "Alice is a young girl who is feeling bored and sleepy. Her sister is sitting on the bank by her, reading a book, but Alice finds it uninteresting because it has no pictures or conversations."
    },
    {
        "question": "What does the White Rabbit take out of its waistcoat-pocket?",
        "ground_truth": "The White Rabbit takes a watch out of its waistcoat-pocket and looks at it before hurrying on."
    }
    ,
    {
        "question": "What is written on the bottle that Alice finds, and what happens when she drinks from it?",
        "ground_truth": "The words 'DRINK ME' are beautifully printed on the bottle in large letters. After drinking it, Alice shrinks down to be only ten inches high."
    },
    {
        "question": "Describe the character of the Caterpillar.",
        "ground_truth": "The Caterpillar is a large, blue caterpillar found sitting on top of a mushroom, smoking a long hookah. It speaks in a sleepy, languid voice and gives Alice short, sometimes cryptic advice."
    },
    {
        "question": "What is the capital of England?",
        "ground_truth": "The story of Alice's Adventures in Wonderland does not contain information about the capital of England."
    },
    {
        "question": "Why did the Mad Hatter and the March Hare put the Dormouse in the teapot?",
        "ground_truth": "The story does not state a clear reason, but they attempt to put the Dormouse in the teapot after the Mad Hatter exclaims that the Dormouse is asleep again and wants to change the subject of conversation."
    }
]
# "question": "What is the capital of England?",
# "ground_truth": "The story of Alice Adventures in Wonderland does not contain information about the capital of England."

# system prompt

# hidden prompts from llama-index

> Notice the question, "What is the capital of England?". This is an out-of-domain question. It tests whether our RAG system correctly uses the provided text and avoids making up answers when the information is not present.

### 4.2 Create the Evaluation Configuration

Next, we'll create a dedicated configuration file for our evaluation scripts.

1.  In the `evaluation` directory, create a new file named `eval_config.py`.
2.  Add the following code to the file.

In [ ]:
from pathlib import Path
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    ResponseRelevancy,
)

# --- Paths for Evaluation ---
EVALUATION_ROOT_PATH: Path = Path('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/eval_config.py') # Path(__file__).parent
EVALUATION_RESULTS_PATH: Path = EVALUATION_ROOT_PATH / "evaluation_results/"
EXPERIMENTAL_VECTOR_STORES_PATH: Path = Path('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/eval_config.py').parent.parent / "local_storage" / "experimental_vector_stores/"
# EXPERIMENTAL_VECTOR_STORES_PATH: Path = Path(__file__).parent.parent / "local_storage" / "experimental_vector_stores/"

# --- Ragas Evaluation Metrics ---
EVAL_METRICS = [
    Faithfulness(),
    # ResponseRelevancy(),
    ContextPrecision(),
    ContextRecall(),
]
EVAL_METRICS

[Faithfulness(_required_columns={<MetricType.SINGLE_TURN: 'single_turn'>: {'response', 'user_input', 'retrieved_contexts'}}, name='faithfulness', llm=None, output_type=<MetricOutputType.CONTINUOUS: 'continuous'>, nli_statements_prompt=NLIStatementPrompt(instruction=Your task is to judge the faithfulness of a series of statements based on a given context. For each statement you must return verdict as 1 if the statement can be directly inferred based on the context or 0 if the statement can not be directly inferred based on the context., examples=[(NLIStatementInput(context='John is a student at XYZ University. He is pursuing a degree in Computer Science. He is enrolled in several courses this semester, including Data Structures, Algorithms, and Database Management. John is a diligent student and spends a significant amount of time studying and completing assignments. He often stays late in the library to work on his projects.', statements=['John is majoring in Biology.', 'John is taking

![Screenshot 2025-09-22 at 20.27.53.png](<attachment:Screenshot 2025-09-22 at 20.27.53.png>)

Here we define the file paths for our evaluation results and experimental vector stores. We also specify the list of metrics from the `ragas` library that we'll use to score our system.

-----

## 5.&nbsp; Building the Evaluation Runner

Now we'll build the main script that runs the entire evaluation process.

In the `evaluation` directory, create a new file named `run_evaluation.py`.

### 5.1 The Imports

First, add all the necessary imports to the top of the empty `run_evaluation.py` file.

In [ ]:
import sys
project_root_dir = "/Users/karimelbana/wbs/rag_project_DSAI/"
sys.path.append(project_root_dir)

In [ ]:
from datetime import datetime

from datasets import Dataset
from llama_index.core import (
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage)

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import BaseQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

import pandas as pd

from ragas import evaluate

from evaluation.eval_config import (
    EVAL_METRICS,
    EVALUATION_RESULTS_PATH,
    EXPERIMENTAL_VECTOR_STORES_PATH,
)

from evaluation.evaluation_questions import EVAL_DATA

from src.config import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    DATA_PATH,
    SIMILARITY_TOP_K,
)
from src.engine.components import get_embedding_model, initialise_llm

### 5.2 Helper Functions

Now, let's add the helper functions that will support our main evaluation logic. We'll add these to `run_evaluation.py` one by one to understand the specific role each one plays in our script.

#### Function 1: Getting the Evaluation Data

Our first helper is a simple function to prepare our questions and answers. Its only job is to read the `EVAL_DATA` list we created earlier and unpack it into two separate lists: one for the questions and one for the ground truth answers.

Add the following function to `run_evaluation.py`:

In [ ]:
def get_eval_data() -> tuple[list[str], list[str]]:
    """Extracts questions and ground truths from the EVAL_DATA constant."""
    return [item["question"] for item in EVAL_DATA], [item["ground_truth"] for item in EVAL_DATA]

In [ ]:
get_eval_data()

(['Who is Alice and what is her sister doing at the beginning of the story?',
  'What does the White Rabbit take out of its waistcoat-pocket?',
  'What is written on the bottle that Alice finds, and what happens when she drinks from it?',
  'Describe the character of the Caterpillar.',
  'What is the capital of England?',
  'Why did the Mad Hatter and the March Hare put the Dormouse in the teapot?'],
 ['Alice is a young girl who is feeling bored and sleepy. Her sister is sitting on the bank by her, reading a book, but Alice finds it uninteresting because it has no pictures or conversations.',
  'The White Rabbit takes a watch out of its waistcoat-pocket and looks at it before hurrying on.',
  "The words 'DRINK ME' are beautifully printed on the bottle in large letters. After drinking it, Alice shrinks down to be only ten inches high.",
  'The Caterpillar is a large, blue caterpillar found sitting on top of a mushroom, smoking a long hookah. It speaks in a sleepy, languid voice and give

#### Function 2: Building or Loading an Experimental Index

Next, we need a function to manage our vector stores for experiments. It's important to keep these separate from the main vector store our chatbot uses. To save time, it will first check if a store already exists. If it does, it loads it; otherwise, it builds a new one.

Add the next function to your file:

![Screenshot 2025-09-22 at 13.34.12.png](<attachment:Screenshot 2025-09-22 at 13.34.12.png>)

In [ ]:
# Load the vector store if exists otherwise build it

def get_or_build_index(
    chunk_size: int,
    chunk_overlap: int,
    embed_model: HuggingFaceEmbedding
) -> VectorStoreIndex:
    """
    Checks for a persisted vector store for this experiment.
    If it exists, it loads it. If not, it builds it, persists it, and returns it.
    """
    vector_store_id: str = f"vs_baseline"
    specific_vector_store_path: Path = EXPERIMENTAL_VECTOR_STORES_PATH / vector_store_id

    if specific_vector_store_path.exists():

        #pointing to index files: knows where is saved index files (docstore, vectorstore, indexstore) via persist_dir
        print(f"--- Loading existing index from: {vector_store_id} ---")
        storage_context: StorageContext = StorageContext.from_defaults(
            persist_dir=str(specific_vector_store_path)
        )

        # rebuilds an in-memory VectorStoreIndex object from already-persisted storage
        # looks inside that persisted folder, reads the stored metadata, docstore, and embeddings,
        # and reconstructs a VectorStoreIndex object in memory.
        # The embed_model is passed in so the index knows how to generate/query embeddings in a consistent way with what was stored
        index: VectorStoreIndex = load_index_from_storage(
            storage_context,
            embed_model=embed_model
        )
    else:
        # Creating Index files from the Alice in Wonderland book
        print(f"--- Creating new index for: {vector_store_id} ---")
        documents: list[Document] = SimpleDirectoryReader(input_dir=DATA_PATH).load_data()

        text_splitter: SentenceSplitter = SentenceSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        index: VectorStoreIndex = VectorStoreIndex.from_documents(
            documents,
            transformations=[text_splitter],
            embed_model=embed_model
        )
        # presisting the index files to hard-desk
        index.storage_context.persist(persist_dir=str(specific_vector_store_path))
        print(f"--- Saved new index to: {vector_store_id} ---")
    return index

#### Function 3: Generating the Question-Answer Dataset

The `ragas` library requires the evaluation data to be in a specific format - a Hugging Face `Dataset` object. This next function handles that process. It iterates through each of our questions, queries the RAG engine to get an answer and the retrieved source context, and then organises this information alongside our ground truths into the required `Dataset` structure.

Add the `generate_qa_dataset` function to your file:

In [ ]:
def generate_qa_dataset(
    query_engine: BaseQueryEngine,
    questions: list[str],
    ground_truths: list[str]
) -> Dataset:
    """
    Generates answers and contexts for a given query engine and returns a HuggingFace Dataset.
    """
    responses: list[str] = []
    contexts: list[list[str]] = []
    for question in questions:
        print(f"Querying for question: '{question[:50]}...'")
        response_object = query_engine.query(question)
        responses.append(str(response_object))
        # the 4 chunks that were retrieved
        contexts.append([node.get_content() for node in response_object.source_nodes])

    response_data: dict[str, list[Any]] = {
        "question": questions,
        "answer": responses,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(response_data)


# def generate_qa_dataset_2(
#     query_engine: BaseQueryEngine,
#     questions: list[str],
#     ground_truths: list[str]
# ) -> Dataset:
#     responses: list[str] = []
#     contexts: list[list[str]] = []

#     for question in questions:
#         response_object = query_engine.query(question)
#         responses.append("" if response_object is None else str(response_object))

#         nodes = getattr(response_object, "source_nodes", [])
#         if not nodes:
#             contexts.append([""])  # placeholder
#         else:
#             contexts.append([str(node.get_content()) for node in nodes])

#     # ensure same length
#     assert len(questions) == len(responses) == len(contexts) == len(ground_truths)

#     response_data = {
#         "question": questions,
#         "answer": responses,
#         "contexts": contexts,
#         "ground_truth": ground_truths,
#     }

#     return Dataset.from_dict(response_data)

#### Function 4: Saving the Results

Finally, we need a utility function to save our results. To make experiment tracking easier, this function saves the output to the `evaluation_results` directory with a timestamp in the filename. It generates two files: a detailed CSV with the score for every single question and a summary CSV that shows the average scores across all questions.

Add this final helper function to `run_evaluation.py`:

In [ ]:
def save_results(
    results_df: pd.DataFrame,
    filename_prefix: str
) -> None:
    """Saves the evaluation results and summary to CSV files."""
    results_dir: Path = EVALUATION_RESULTS_PATH # '/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results'
    print("EVALUATION_RESULTS_PATH", f'{EVALUATION_RESULTS_PATH}')
    results_dir.mkdir(exist_ok=True, parents=True)
    timestamp: str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    detailed_path: Path = results_dir / f"{filename_prefix}_detailed_{timestamp}.csv"
    # the detailed csv
    results_df.to_csv(detailed_path, index=False)
    print(f"\\n--- 💾 Detailed results saved to {detailed_path} ---")

    summary_path: Path = results_dir / f"{filename_prefix}_summary_{timestamp}.csv"
    param_cols: list[str] = [
        col
        for col in ['chunk_size', 'chunk_overlap']
        if col in results_df.columns
    ]
    if param_cols:
        # the summary csv
        avg_scores: pd.DataFrame = results_df.groupby(param_cols).mean(numeric_only=True)
        avg_scores.to_csv(summary_path)
        print(f"--- 💾 Summary of average scores saved to {summary_path} ---")

### 5.3 The Main Evaluation Function

This function orchestrates the evaluation. It uses our helper functions to get the data, build a query engine, generate answers, run the `ragas` evaluation, and save the results.

Add the following function to `run_evaluation.py`.

In [ ]:
print(qa_dataset)
print(qa_dataset[0])

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 6
})
{'question': 'Who is Alice and what is her sister doing at the beginning of the story?', 'answer': "Alice is the main character in the story. At the beginning of the story, Alice's sister is watching the setting sun and thinking of Alice and her wonderful adventures.", 'contexts': ['“Wake up, Alice dear!” said her sister; “Why, what a long sleep you’ve\r\nhad!”\r\n\r\n“Oh, I’ve had such a curious dream!” said Alice, and she told her\r\nsister, as well as she could remember them, all these strange\r\nAdventures of hers that you have just been reading about; and when she\r\nhad finished, her sister kissed her, and said, “It _was_ a curious\r\ndream, dear, certainly: but now run in to your tea; it’s getting late.”\r\nSo Alice got up and ran off, thinking while she ran, as well she might,\r\nwhat a wonderful dream it had been.\r\n\r\n\r\nBut her sister sat still just as she left her, leaning her 

In [ ]:
def evaluate_baseline(
    llm: Groq,
    embed_model: HuggingFaceEmbedding
) -> pd.DataFrame:
    """Evaluates the RAG system using the baseline settings from config.py."""
    print("\n--- 🚀 Stage 1: Evaluating Baseline Configuration ---")
    questions, ground_truths = get_eval_data()

    index: VectorStoreIndex = get_or_build_index(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, embed_model=embed_model
    )
    # chat engine from the vector DB to get the responses
    query_engine: BaseQueryEngine = index.as_query_engine(similarity_top_k=SIMILARITY_TOP_K, llm=llm)

    qa_dataset: Dataset = generate_qa_dataset(query_engine, questions, ground_truths)

    print("--- Running Ragas evaluation for baseline... ---")
    result: Dataset = evaluate(
        dataset=qa_dataset,
        metrics=EVAL_METRICS,
        raise_exceptions=True,
    )

    results_df: pd.DataFrame = result.to_pandas()
    results_df['chunk_size'] = CHUNK_SIZE
    results_df['chunk_overlap'] = CHUNK_OVERLAP

    save_results(results_df, "baseline_evaluation")
    print("--- ✅ Baseline Evaluation Complete ---")
    return results_df

### 5.4 The Execution Block

Finally, add the standard `if __name__ == "__main__"` block to make our script runnable. This will initialise the LLM and embedding models and start the evaluation process.

Add this final block of code to the end of `run_evaluation.py`.

In [ ]:
if __name__ == "__main__":
    llm_to_test: Groq = initialise_llm()
    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    evaluate_baseline(llm=llm_to_test, embed_model=embed_model_to_test)


--- 🚀 Stage 1: Evaluating Baseline Configuration ---
--- Loading existing index from: vs_baseline ---
Loading llama_index.core.storage.kvstore.simple_kvstore from /Users/karimelbana/wbs/rag_project_DSAI/local_storage/experimental_vector_stores/vs_baseline/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /Users/karimelbana/wbs/rag_project_DSAI/local_storage/experimental_vector_stores/vs_baseline/index_store.json.
Querying for question: 'Who is Alice and what is her sister doing at the b...'
Querying for question: 'What does the White Rabbit take out of its waistco...'
Querying for question: 'What is written on the bottle that Alice finds, an...'
Querying for question: 'Describe the character of the Caterpillar....'
Querying for question: 'What is the capital of England?...'
Querying for question: 'Why did the Mad Hatter and the March Hare put the ...'
--- Running Ragas evaluation for baseline... ---


Evaluating: 100%|██████████| 18/18 [00:38<00:00,  2.15s/it]


NotADirectoryError: [Errno 20] Not a directory: '/Users/karimelbana/wbs/rag_project_DSAI/evaluation/eval_config.py/evaluation_results'

In [ ]:
import pandas as pd

# Make sure to replace the timestamp with the one from your actual results file
# results_df = pd.read_csv(project_root_dir + "/evaluation/evaluation_results/baseline_evaluation_detailed_2025-09-22_18-37-01.csv")
# results_df = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/baseline_evaluation_detailed_2025-09-23_11-16-44.csv')
results_df = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/baseline_evaluation_summary_2025-09-23_11-16-44.csv')

results_df.head()

,chunk_size,chunk_overlap,faithfulness,context_precision,context_recall
0,512,80,0.590909,0.430556,0.333333


Your `run_evaluation.py` file is now complete.

-----

## 6.&nbsp; Run the Evaluation

With all the files in place, you're ready to run the evaluation.

From your VSCode terminal, run the following command:

In [ ]:
# python -m evaluation.run_evaluation

The script will print its progress to the terminal. The first time it runs, it will create and save a new vector store, which may take a moment. Subsequent runs will load the existing store instantly.

-----

## 7.&nbsp; Interpret the Results

Once the script finishes, you will find a new `evaluation_results` folder inside your `evaluation` directory. This folder contains two CSV files with timestamps in their names:

1.  **`baseline_evaluation_detailed_...csv`**: This file contains the scores for each individual question across all the metrics.
2.  **`baseline_evaluation_summary_...csv`**: This file shows the average score for each metric across all questions, giving you a high-level view of the system's performance.

Open the summary file. You will see the average scores for our four key metrics:

* **faithfulness**: Measures if the answer is factually consistent with the retrieved context. A low score here indicates hallucination.
* **response_relevancy**: Measures how relevant the answer is to the question. A low score means the answer is off-topic.
* **context_precision**: Measures the signal-to-noise ratio of the retrieved context. A low score means many of the retrieved chunks were irrelevant.
* **context_recall**: Measures if all the necessary information to answer the question was retrieved. A low score means the retriever missed important context.

-----

## 8.&nbsp; Challenges and Further Exploration 😀

You've now learned how to build a complete evaluation pipeline using our "Alice in Wonderland" example. The final and most important step is to apply these techniques to the custom RAG chatbot you've been building.

These challenges will guide you through creating an evaluation framework for your own project, establishing its baseline performance, and digging into the results to find opportunities for improvement.

### 1. Build the Evaluation Framework for Your RAG Chatbot

It's time to create the "scorecard" for your project. This involves creating a custom "answer key" with ground truths for your specific document and adapting the evaluation script to test your pipeline.

**Your Mission:**

1.  Write Your Ground Truths: Write 5-7 challenging questions about the document you used in your RAG pipeline. Most importantly, you must also write the detailed, factually correct `ground_truth` answers for each. This is a crucial skill for any AI engineer!
2.  Run Your Baseline Evaluation: Execute the script from your terminal. It'll now use your questions to test your RAG pipeline. When it's finished, open the `_summary.csv` file. What is the baseline score for your custom RAG system?

### 2. Analysing Your Results in a Notebook

Aggregate scores are useful, but the most valuable insights come from analysing individual failures. For this, we'll use a familiar friend - a Notebook. It's time to put on your detective hat and investigate your own project's results.

**Your Mission:**

1.  Set Up Your Analysis Environment:

      * In your project's root directory (at the same level as `src` and `evaluation`), create a new folder named `notebooks`. This is the standard place to keep your exploratory analysis separate from your application code.
      * Inside the new `notebooks` folder, create a new Notebook file. Name it `01_baseline_analysis.ipynb`. This clear naming scheme, with a numerical prefix, helps keep your experiments organised chronologically.

2.  **Load and Explore Your Results:**

      * Open your new notebook. In the first cell, you'll need to load the detailed results CSV file you generated. You'll need to replace the timestamped part of the filename with the one your script created.

    ```python
    import pandas as pd

    # Make sure to replace the timestamp with the one from your actual results file
    results_df = pd.read_csv("../evaluation/evaluation_results/baseline_evaluation_detailed_2025-06-07_13-10-00.csv")

    results_df.head()
    ```

3.  **Structure Your Investigation:**

      * Use Markdown cells and headings (`##`, `###`) to structure your analysis. A well-organised notebook is a key professional skill, as it makes your work easy for others (and your future self) to understand. Your notebook should tell a clear story.

4.  **Find and Analyse a Failure:**

      * In your notebook, find the question with the lowest score for a specific metric (e.g., the lowest `faithfulness` or `context_recall` score).
      * Create a new section in your notebook for that metric (e.g., `## Investigating Low Faithfulness`).
      * In that section, write a clear analysis explaining why you think the score was low. Use the data from the DataFrame (`question`, `contexts`, `answer`, and `ground_truth`) to support your conclusion.

This process of structured error analysis in a clean, well-documented notebook is one of the most important skills for building reliable and high-performing AI systems.